In [1]:
from vertex_lite.custom import load_data_v3 as load_data
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 
import glob
import os
import numpy as np
import pyprojroot.here as here

In [2]:
def filter_data(time, local=False, cluster=False):
    rpath = here('review/resultados_cluster' if cluster else 'review/resultados_random')

    df = load_data(
        f'{rpath}/noise_*/pct_*/sim_*/cells.parquet',
        columns=['time', 'cell_id', 'neighbor_of_mutant', 'alive', 'mutant', 't_eff'],
        time_filter=time,
    )

    if df.empty:
        print(f'No data found for time {time}')
        return pd.DataFrame()

    df = df[df['alive'] == 1].copy()
    if local:
        df = df[df['neighbor_of_mutant'] == True].copy()
        if 'pct_mutant' in df.columns and not (df['pct_mutant'] == 0).any():
            # Get all unique mesh types so we can create a full row of dummy values
            mesh_types = df['mesh_type'].unique() if 'mesh_type' in df.columns else [None]
            
            dummy_rows = [{
                'time': time,
                'cell_id': None,
                'neighbor_of_mutant': True,
                'alive': 1,
                'mutant': 0,
                't_eff': 0.0,
                'pct_mutant': 0.0,
                'mesh_type': mt
            } for mt in mesh_types]
            
            df = pd.concat([df, pd.DataFrame(dummy_rows)], ignore_index=True)

    return df

In [3]:
# def calculate_teff_stats(df):
#     groupby_cols = ['time','mesh_type','pct_mutant']
#     teff_stats = df.groupby(groupby_cols).apply(
#         lambda x: pd.Series({
#             'teff_mutant': x[x['mutant'] == 1]['t_eff'].mean(),
#             'teff_mutant_std': x[x['mutant'] == 1]['t_eff'].std(),
#             'teff_mutant_sem': x[x['mutant'] == 1]['t_eff'].sem(),
#             'teff_wildtype': x[x['mutant'] == 0]['t_eff'].mean(),
#             'teff_wildtype_std': x[x['mutant'] == 0]['t_eff'].std(),
#             'teff_wildtype_sem': x[x['mutant'] == 0]['t_eff'].sem(),
#             'teff_global': x['t_eff'].mean(),
#             'teff_global_std': x['t_eff'].std(),
#             'teff_global_sem': x['t_eff'].sem(),
#             'n_mutant': x[x['mutant'] == 1].shape[0],
#             'n_wildtype': x[x['mutant'] == 0].shape[0],
#             'n_total': x.shape[0],
#         })
#         , include_groups=False
#     ).reset_index()
#     return teff_stats

In [4]:
def calculate_teff_stats(df):
    if df.empty:
        return pd.DataFrame()

    groupby_cols = ['time', 'mesh_type', 'pct_mutant']
    return df.groupby(groupby_cols).apply(
        lambda group: pd.Series({
            'teff_mutant': group.loc[group['mutant'] == 1, 't_eff'].mean(),
            'teff_wildtype': group.loc[group['mutant'] == 0, 't_eff'].mean(),
            'teff_all': group['t_eff'].mean(),
        }),
        include_groups=False,
    ).reset_index()


def get_heatmap_data(time, local=False, case='all', cluster=False):
    df = filter_data(time, local=local, cluster=cluster)
    stats = calculate_teff_stats(df)

    if stats.empty:
        return pd.DataFrame(), f'Teff {case}'

    value_column = f'teff_{case}'
    if value_column not in stats.columns:
        raise ValueError("case must be 'all', 'mutant', or 'wildtype'")

    heatmap_data = (
        stats.pivot(index='pct_mutant', columns='mesh_type', values=value_column)
        .sort_index()
        .sort_index(axis=1)
    )
    return heatmap_data, f'Teff {case}'


import matplotlib.colors as mcolors
from matplotlib import colormaps

def plot_heatmap(heatmap_data, title, time, local=False, case='all', cluster=False):
    if heatmap_data.empty:
        print(f'No heatmap data available for time {time}, case {case}')
        return None

    finite_values = heatmap_data.to_numpy()
    finite_values = finite_values[np.isfinite(finite_values) & (finite_values > 0)]  # Filter out non-finite and non-positive values
    if finite_values.size == 0:
        print(f'No finite Teff values available for time {time}, case {case}, local={local}')
        return None

    cmap = colormaps['gnuplot']
    # value_min = float(finite_values.min())
    # value_max = float(finite_values.max())
    # norm = mcolors.Normalize(vmin=0.33, vmax=0.39) if value_min != value_max else None
    fixed_vmin = 0.25
    fixed_vmax = 1.0
    norm = mcolors.Normalize(vmin=fixed_vmin, vmax=fixed_vmax)

    sns.set_theme(style='white', context='talk')
    fig, ax = plt.subplots(figsize=(7, 5))
    im = ax.imshow(
        heatmap_data.values,
        cmap=cmap,
        origin='lower',
        aspect='auto',
        norm=norm,
    )

    step = max(1, len(heatmap_data.columns) // 8)
    x_ticks = np.arange(0, len(heatmap_data.columns), step)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels(heatmap_data.columns[::step], rotation=45)

    y_step = max(1, len(heatmap_data.index) // 8)
    y_ticks = np.arange(0, len(heatmap_data.index), y_step)
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(heatmap_data.index[::y_step])

    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label('Teff')
    ax.set_xlabel('Initial Mesh Distortion')
    ax.set_ylabel('% mutants')
    ax.set_title(f'{title} for Neighbors of Mutant Cells' if local else f'{title} for Whole Tissue')
    ax.annotate(f'T={time} | local={local}', xy=(0.0, -0.2), xycoords='axes fraction', ha='left', fontsize=10)

    savedir = here('review/analysis/teff/plots/heatmaps/pilar')
    os.makedirs(savedir, exist_ok=True)
    clustering = 'cluster' if cluster else 'random'
    suffix = 'local' if local else 'global'
    filename = f'{savedir}/heatmap_{clustering}_time_{time}_{case}_{suffix}.svg'
    fig.savefig(filename, format='svg', bbox_inches='tight')
    plt.close(fig)
    return filename

In [5]:
def get_difference_heatmap_data(end_time, local=False, case='all', cluster=False):
    # Fetch data only at the requested time
    df = filter_data(end_time, local=local, cluster=cluster)
    stats = calculate_teff_stats(df)

    if stats.empty:
        return pd.DataFrame(), f'Teff Difference from 0% mutants ({end_time}) {case}'

    value_column = f'teff_{case}'
    if value_column not in stats.columns:
        raise ValueError("case must be 'all', 'mutant', or 'wildtype'")

    heatmap_data = (
        stats.pivot(
            index='pct_mutant',
            columns='mesh_type',
            values=value_column,
        )
        .sort_index()
        .sort_index(axis=1)
    )

    if 0 not in heatmap_data.index:
        return pd.DataFrame(), f'Teff Difference from 0% mutants ({end_time}) {case}'

    # Subtract the corresponding mesh_type value at pct_mutant = 0
    baseline = heatmap_data.loc[0]
    diff_data = heatmap_data.subtract(baseline, axis='columns')

    return diff_data, f'Teff Difference from 0% mutants ({end_time}) {case}'


def plot_difference_heatmap(heatmap_data, title, end_time, local=False, case='all', cluster=False):
    if heatmap_data.empty:
        print(f'No difference data available for time {end_time}, case {case}')
        return None

    finite_values = heatmap_data.to_numpy()
    finite_values = finite_values[np.isfinite(finite_values)]
    if finite_values.size == 0:
        return None

    # Use a diverging colormap and symmetric normalization centered at 0
    cmap = sns.color_palette('coolwarm', as_cmap=True)
    vmax = max(abs(finite_values.min()), abs(finite_values.max()))
    norm = mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax) if vmax != 0 else None

    sns.set_theme(style='white', context='talk')
    fig, ax = plt.subplots(figsize=(7, 5))
    im = ax.imshow(
        heatmap_data.values,
        cmap=cmap,
        origin='lower',
        aspect='auto',
        norm=norm,
    )

    step = max(1, len(heatmap_data.columns) // 8)
    ax.set_xticks(np.arange(0, len(heatmap_data.columns), step))
    ax.set_xticklabels(heatmap_data.columns[::step], rotation=45)

    y_step = max(1, len(heatmap_data.index) // 8)
    ax.set_yticks(np.arange(0, len(heatmap_data.index), y_step))
    ax.set_yticklabels(heatmap_data.index[::y_step])

    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label('Δ Teff (End - 0)')
    ax.set_xlabel('Initial Mesh Distortion')
    ax.set_ylabel('% mutants')
    ax.set_title(f'{title} ({case})')
    ax.annotate(f'T={end_time} minus T=0 | local={local}', xy=(0.0, -0.2), xycoords='axes fraction', ha='left', fontsize=10)

    savedir = here('review/analysis/teff/plots/heatmaps/difference')
    os.makedirs(savedir, exist_ok=True)
    clustering = 'cluster' if cluster else 'random'
    suffix = 'local' if local else 'global'
    filename = f'{savedir}/heatmap_diff_{clustering}_{end_time}_{case}_{suffix}.svg'
    fig.savefig(filename, format='svg', bbox_inches='tight')
    plt.close(fig)
    return filename

In [6]:
times = {
    False: [0,14980],
    True: [0,14980],
}

for case in ['all']:
    for local in [True, False]:
        for cluster in [True, False]:
            for time in times[cluster]:
                print(f'Plotting case={case}, local={local}, cluster={cluster}, time={time}')
                heatmap_data, title = get_heatmap_data(
                    time=time,
                    local=local,
                    case=case,
                    cluster=cluster,
                )
                output_file = plot_heatmap(
                    heatmap_data,
                    title,
                    time=time,
                    local=local,
                    case=case,
                    cluster=cluster,
                )
                if output_file:
                    print(f'Saved {output_file}')

print('All Teff heatmaps saved.')

Plotting case=all, local=True, cluster=True, time=0
Saved /home/vmanso/Victor/Tissue Characterization Paper/TFM-Vertex-Model/review/analysis/teff/plots/heatmaps/pilar/heatmap_cluster_time_0_all_local.svg
Plotting case=all, local=True, cluster=True, time=14980
Saved /home/vmanso/Victor/Tissue Characterization Paper/TFM-Vertex-Model/review/analysis/teff/plots/heatmaps/pilar/heatmap_cluster_time_14980_all_local.svg
Plotting case=all, local=True, cluster=False, time=0
Saved /home/vmanso/Victor/Tissue Characterization Paper/TFM-Vertex-Model/review/analysis/teff/plots/heatmaps/pilar/heatmap_random_time_0_all_local.svg
Plotting case=all, local=True, cluster=False, time=14980
Saved /home/vmanso/Victor/Tissue Characterization Paper/TFM-Vertex-Model/review/analysis/teff/plots/heatmaps/pilar/heatmap_random_time_14980_all_local.svg
Plotting case=all, local=False, cluster=True, time=0
Saved /home/vmanso/Victor/Tissue Characterization Paper/TFM-Vertex-Model/review/analysis/teff/plots/heatmaps/pilar/

In [7]:
times = {
    False: 14980,
    True: 14980,
}

for case in ['all']:
    for local in [True, False]:
        for cluster in [True, False]:
            time = times[cluster]
            print(f'Plotting difference for case={case}, local={local}, cluster={cluster}, time={time}')
            diff_data, title = get_difference_heatmap_data(
                end_time=time,
                local=local,
                case=case,
                cluster=cluster,
            )
            output_file = plot_difference_heatmap(
                diff_data,
                title,
                end_time=time,
                local=local,
                case=case,
                cluster=cluster,
            )
            if output_file:
                print(f'Saved {output_file}')

print('All Teff difference heatmaps saved.')

Plotting difference for case=all, local=True, cluster=True, time=14980
Saved /home/vmanso/Victor/Tissue Characterization Paper/TFM-Vertex-Model/review/analysis/teff/plots/heatmaps/difference/heatmap_diff_cluster_14980_all_local.svg
Plotting difference for case=all, local=True, cluster=False, time=14980
Saved /home/vmanso/Victor/Tissue Characterization Paper/TFM-Vertex-Model/review/analysis/teff/plots/heatmaps/difference/heatmap_diff_random_14980_all_local.svg
Plotting difference for case=all, local=False, cluster=True, time=14980
Saved /home/vmanso/Victor/Tissue Characterization Paper/TFM-Vertex-Model/review/analysis/teff/plots/heatmaps/difference/heatmap_diff_cluster_14980_all_global.svg
Plotting difference for case=all, local=False, cluster=False, time=14980
Saved /home/vmanso/Victor/Tissue Characterization Paper/TFM-Vertex-Model/review/analysis/teff/plots/heatmaps/difference/heatmap_diff_random_14980_all_global.svg
All Teff difference heatmaps saved.
